In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_as_bert = AutoModel.from_pretrained(MODEL_NAME)


In [2]:
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


In [11]:
# Sentences we want sentence embeddings for
sentence_0 = ['千葉県<pad>千葉市']

# Tokenize sentences
sentence_tokenized_0 = tokenizer(sentence_0, padding=True, truncation=True, return_tensors='pt')
sentence_tokens_0 = tokenizer.convert_ids_to_tokens(sentence_tokenized_0['input_ids'][0])

print(sentence_tokenized_0)
print(sentence_tokens_0)

{'input_ids': tensor([[    0,     6,  2825, 16171,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}
['<s>', '▁', '光', '電話', '</s>']


In [4]:
# Compute token embeddings
with torch.no_grad():
    bert_output_0 = model_as_bert(**sentence_tokenized_0)

print(bert_output_0)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.0166,  0.0908, -0.0131,  ...,  0.0101,  0.0535, -0.0925],
         [ 0.0289,  0.0084, -0.0141,  ..., -0.1118,  0.1631, -0.2405],
         [ 0.0310,  0.0384, -0.0157,  ..., -0.0095,  0.1424, -0.1085],
         ...,
         [ 0.0154,  0.0836, -0.0154,  ...,  0.0102,  0.0931, -0.0756],
         [-0.0009,  0.1238, -0.0156,  ..., -0.0494,  0.1087, -0.1761],
         [ 0.0601,  0.0640, -0.0134,  ..., -0.0513,  0.1796, -0.1428]]]), pooler_output=tensor([[-4.5999e-03,  6.9184e-02,  5.2132e-02,  3.0980e-03, -3.3168e-02,
          2.5383e-02, -5.9955e-03,  1.9132e-02, -7.5103e-04, -8.9024e-02,
         -2.2890e-02, -4.9924e-02, -7.3825e-04,  5.5216e-02,  7.9394e-02,
          3.3648e-02, -1.4329e-02,  1.4247e-03, -2.0411e-04, -1.3845e-02,
          1.0009e-03,  2.5500e-02, -1.4870e-02, -1.9685e-02,  3.4898e-04,
          5.0260e-04,  4.5579e-03, -4.7979e-02, -1.9190e-02,  1.6330e-02,
         -3.4830e-02,  6.3043e-03,  

In [5]:
token_embeddings_0 = bert_output_0[0]
print(token_embeddings_0.size())
print(token_embeddings_0)

torch.Size([1, 9, 768])
tensor([[[-0.0166,  0.0908, -0.0131,  ...,  0.0101,  0.0535, -0.0925],
         [ 0.0289,  0.0084, -0.0141,  ..., -0.1118,  0.1631, -0.2405],
         [ 0.0310,  0.0384, -0.0157,  ..., -0.0095,  0.1424, -0.1085],
         ...,
         [ 0.0154,  0.0836, -0.0154,  ...,  0.0102,  0.0931, -0.0756],
         [-0.0009,  0.1238, -0.0156,  ..., -0.0494,  0.1087, -0.1761],
         [ 0.0601,  0.0640, -0.0134,  ..., -0.0513,  0.1796, -0.1428]]])


In [6]:
# Perform pooling. In this case, max pooling.
sentence_embeddings_0 = mean_pooling(bert_output_0, sentence_tokenized_0['attention_mask'])

print(sentence_embeddings_0)

tensor([[ 2.2249e-02,  5.4540e-02, -1.5351e-02,  8.5744e-03,  1.4341e-01,
         -2.9087e-03,  4.2510e-02,  1.4572e-02, -4.0439e-04,  6.2067e-02,
         -5.0681e-02, -4.7251e-02,  8.4635e-02, -4.4659e-02,  1.2939e-02,
         -3.4484e-01,  2.1827e-02,  1.6846e-01, -1.6644e-02,  8.1302e-03,
          6.3902e-03, -2.5572e-02,  9.2607e-02, -5.9916e-03,  2.9066e-03,
         -2.4836e-02, -1.8462e-03, -6.5301e-02,  2.1418e-02, -9.0934e-03,
          1.3128e-01, -1.3705e-01,  4.1440e-02,  1.2313e-01,  1.7877e-02,
         -3.4780e-02, -4.8206e-02, -5.4489e-02, -5.1392e-02,  9.0665e-03,
          1.5354e-01, -2.8625e-02, -7.0340e-02,  6.8061e-02, -8.2545e-02,
          8.7678e-02,  1.2499e-01,  9.7091e-02,  1.2361e-01,  6.5309e-02,
          2.7503e-02, -2.5413e-02, -1.5953e-01,  4.7345e-02, -1.7379e-01,
         -2.3690e-02, -1.3114e-01, -1.1040e-01, -1.5588e-01,  1.3487e-01,
         -3.7196e-03, -9.6621e-02,  2.2893e-02, -6.1784e-02, -1.2368e-01,
         -2.3435e-02,  5.5251e-02,  5.

In [8]:
attention_mask_0 = sentence_tokenized_0['attention_mask']
# attention_mask = torch.Tensor([[1, 0, 1, 1, 1, 1, 1, 1]])

print(attention_mask_0)
print(attention_mask_0.size())
print(attention_mask_0.unsqueeze(-1))
print(attention_mask_0.unsqueeze(-1).expand(token_embeddings_0.size()))
print(attention_mask_0.unsqueeze(-1).expand(token_embeddings_0.size()).float())

mask_expanded_0 = attention_mask_0.unsqueeze(-1).expand(token_embeddings_0.size()).float()
print(torch.sum(token_embeddings_0 * mask_expanded_0, 1))

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([1, 9])
tensor([[[1],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1]]])
tensor([[[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]])
tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]])
tensor([[ 2.0024e-01,  4.9086e-01, -1.3816e-01,  7.7169e-02,  1.2907e+00,
         -2.6178e-02,  3.8259e-01,  1.3115e-01, -3.6395e-03,  5.5861e-01,
         -4.5613e-01, -4.2526e-01,  7.6172e-01, -4.0193e-01,  1.1645e-01,
         -3.1036e+00,  1.9644e-01,  1.5162e+00, -1.4979e-01,  7.3172e-02,
          5.7512e-02, -2.3015e-

In [9]:
print(torch.sum(token_embeddings_0, 1))

tensor([[ 2.0024e-01,  4.9086e-01, -1.3816e-01,  7.7169e-02,  1.2907e+00,
         -2.6178e-02,  3.8259e-01,  1.3115e-01, -3.6395e-03,  5.5861e-01,
         -4.5613e-01, -4.2526e-01,  7.6172e-01, -4.0193e-01,  1.1645e-01,
         -3.1036e+00,  1.9644e-01,  1.5162e+00, -1.4979e-01,  7.3172e-02,
          5.7512e-02, -2.3015e-01,  8.3346e-01, -5.3924e-02,  2.6159e-02,
         -2.2352e-01, -1.6615e-02, -5.8771e-01,  1.9276e-01, -8.1841e-02,
          1.1815e+00, -1.2335e+00,  3.7296e-01,  1.1082e+00,  1.6089e-01,
         -3.1302e-01, -4.3385e-01, -4.9040e-01, -4.6253e-01,  8.1599e-02,
          1.3819e+00, -2.5763e-01, -6.3306e-01,  6.1255e-01, -7.4291e-01,
          7.8911e-01,  1.1249e+00,  8.7382e-01,  1.1125e+00,  5.8778e-01,
          2.4753e-01, -2.2872e-01, -1.4357e+00,  4.2610e-01, -1.5642e+00,
         -2.1321e-01, -1.1803e+00, -9.9359e-01, -1.4029e+00,  1.2139e+00,
         -3.3477e-02, -8.6959e-01,  2.0604e-01, -5.5606e-01, -1.1131e+00,
         -2.1091e-01,  4.9725e-01,  4.

In [10]:
print(torch.norm(sentence_embeddings_0))

tensor(2.2827)


In [ ]:
# Sentences we want sentence embeddings for
query = ['千葉県流山市']
question_1 = ['千葉県流山市の']
question_2 = ['流山']


# Tokenize sentences
query_tokenized = tokenizer(query, padding=True, truncation=True, return_tensors='pt')
question_tokenized_1 = tokenizer(question_1, padding=True, truncation=True, return_tensors='pt')
question_tokenized_2 = tokenizer(question_2, padding=True, truncation=True, return_tensors='pt')

query_tokens = tokenizer.convert_ids_to_tokens(query_tokenized['input_ids'][0])
question_tokens_1 = tokenizer.convert_ids_to_tokens(question_tokenized_1['input_ids'][0])
question_tokens_2 = tokenizer.convert_ids_to_tokens(question_tokenized_2['input_ids'][0])

print(query_tokens)
print(question_tokens_1)
print(question_tokens_2)

In [20]:
query_attention_mask = query_tokenized['attention_mask']
question_attention_mask_1 = question_tokenized_1['attention_mask']
question_attention_mask_2 = question_tokenized_2['attention_mask']

print(query_attention_mask)
print(question_attention_mask_1)
print(question_attention_mask_2)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1]])


In [39]:
# reset attention masks
query_weighted_attention_mask = torch.Tensor([[1, 1, 1, 0, 0, 1, 1, 1, 1]])
question_weighted_attention_mask_1 = torch.Tensor([[1, 1, 1, 0, 0, 1, 1]])
question_weighted_attention_mask_2 = torch.Tensor([[1, 1, 1, 1, 1, 1]])

print(query_weighted_attention_mask)
print(question_weighted_attention_mask_1)
print(question_weighted_attention_mask_2)

tensor([[1., 1., 1., 0., 0., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 0., 0., 1., 1.]])
tensor([[1., 1., 1., 1., 1., 1.]])


In [19]:
def weighted_mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


In [13]:
# Compute token embeddings.
with torch.no_grad():
    query_berted = model_as_bert(**query_tokenized)
    question_berted_1 = model_as_bert(**question_tokenized_1)
    question_berted_2 = model_as_bert(**question_tokenized_2)


In [40]:
# Perform weighted pooling.
query_embedding = weighted_mean_pooling(query_berted, query_weighted_attention_mask)
question_embedding_1 = weighted_mean_pooling(question_berted_1, question_weighted_attention_mask_1)
question_embedding_2 = weighted_mean_pooling(question_berted_2, question_weighted_attention_mask_2)

# print(query_embedding)
# print(question_embedding_1)
# print(question_embedding_2)

In [41]:
from sentence_transformers import util

print(util.cos_sim(query_embedding, question_embedding_1)[0][0])
print(util.cos_sim(query_embedding, question_embedding_2)[0][0])

tensor(0.9607)
tensor(0.9729)


In [ ]:
# 1/1
# tensor(0.9677)
# tensor(0.9748)

# 1/2
# tensor(0.9648)
# tensor(0.9744)

# 1/3
# tensor(0.9636)
# tensor(0.9740)

# 1/100
# tensor(0.9608)
# tensor(0.9729)

# 0
# tensor(0.9607)
# tensor(0.9729)